In [1]:
import numpy as np
import uuid
from PIL import Image
from tqdm import tqdm 
from collections import Counter
import re

import os
import sys
import random
from tqdm import tqdm 
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:

path_to_hovernet = "/media/nas/MP/segmentation/hovernet/"
PNG_Paddy_7 = "PNG_Paddy07-2024/"

# Search for all subfolders (classes) in "PNG_Paddy07-2024/"
def find_subfolders(base_dir):
        subfolders = []
        for root, dirs, files in os.walk(base_dir): 
            for dir in dirs:
                subfolders.append(os.path.join(root, dir))
        return subfolders

all_subfolders = find_subfolders(path_to_hovernet+PNG_Paddy_7)
len(all_subfolders), all_subfolders[:]

(7,
 ['/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_PADDY-1509_HARYANA_3-7-24',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_PADDY_1121_HARYANA_8-7-2024',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_1718',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_SHARBATI',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_SUGANDHA_HARYANA 3-7-24',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA HARYANA',
  '/media/nas/MP/segmentation/hovernet/PNG_Paddy07-2024/PUSA_PADDY_1401_HARYANA 3-7-24'])

In [5]:
os.path.basename(all_subfolders[0])

'PUSA_PADDY-1509_HARYANA_3-7-24'

In [6]:
# Find all npz files for each class 
def find_npz_files(base_dirs):
    npz_dict = {}

    for base_dir in base_dirs:
        folder_name = os.path.basename(base_dir)
        for root, dirs, files in os.walk(base_dir):  # Recursively walk through subdirectories
            for file in files:
                if file.endswith('.npz'):  # Check if the file ends with .npz
                    npz_dict.setdefault(folder_name, []).append(os.path.join(root, file))

    return npz_dict

all_npz_dict = find_npz_files(all_subfolders)

In [7]:
npz_counts_per_folder = {key: len(value) for key, value in all_npz_dict.items()}
print(npz_counts_per_folder)

{'PUSA_PADDY-1509_HARYANA_3-7-24': 30, 'PUSA_PADDY_1121_HARYANA_8-7-2024': 23, 'PUSA_1718': 11, 'PUSA_SHARBATI': 10, 'PUSA_SUGANDHA_HARYANA 3-7-24': 21, 'PUSA HARYANA': 21, 'PUSA_PADDY_1401_HARYANA 3-7-24': 26}


In [8]:
max_samples = max(npz_counts_per_folder.values())
mean = np.mean(list(npz_counts_per_folder.values()))
median = np.median(list(npz_counts_per_folder.values()))
max_samples, mean, median

(30, np.float64(20.285714285714285), np.float64(21.0))

In [9]:
lower_bound = min(npz_counts_per_folder.values())
upper_bound = max(npz_counts_per_folder.values())
lower_bound, upper_bound

(10, 30)

In [10]:
sampled_npz_dict_7class = {key: random.sample(value, min(lower_bound, upper_bound)) for key, value in all_npz_dict.items()}

In [11]:
{key: len(value) for key, value in sampled_npz_dict_7class.items()}

{'PUSA_PADDY-1509_HARYANA_3-7-24': 10,
 'PUSA_PADDY_1121_HARYANA_8-7-2024': 10,
 'PUSA_1718': 10,
 'PUSA_SHARBATI': 10,
 'PUSA_SUGANDHA_HARYANA 3-7-24': 10,
 'PUSA HARYANA': 10,
 'PUSA_PADDY_1401_HARYANA 3-7-24': 10}

In [12]:
# Flatten the dictionary into a list of (Class, File Path)
data = [(class_name, file_path) for class_name, files in sampled_npz_dict_7class.items() for file_path in files]

df_samples = pd.DataFrame(data, columns=["Class", "File Path"])

In [13]:
df_samples.to_csv("track_7_class/sampled_files_per_class_7_class.csv", index=False)

In [14]:
df_samples['Class'].value_counts()

Class
PUSA_PADDY-1509_HARYANA_3-7-24      10
PUSA_PADDY_1121_HARYANA_8-7-2024    10
PUSA_1718                           10
PUSA_SHARBATI                       10
PUSA_SUGANDHA_HARYANA 3-7-24        10
PUSA HARYANA                        10
PUSA_PADDY_1401_HARYANA 3-7-24      10
Name: count, dtype: int64

In [15]:
print(len(df_samples))

70


In [16]:
train_df, temp_df = train_test_split(df_samples, test_size=0.3, stratify=df_samples['Class'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Class'], random_state=42)

In [17]:
track = "track_7_class"
os.makedirs(track, exist_ok=True)
train_df.to_csv(f"{track}/train7class.csv", index=False)
val_df.to_csv(f"{track}/val7class.csv", index=False) 
test_df.to_csv(f"{track}/test7class.csv", index=False)

In [20]:
PNG_Paddy_7Class = "PNG_Paddy07-2024-7Class/"

In [21]:
os.makedirs(f"{PNG_Paddy_7Class}/train", exist_ok=True)
os.makedirs(f"{PNG_Paddy_7Class}/val", exist_ok=True)
os.makedirs(f"{PNG_Paddy_7Class}/test", exist_ok=True)

In [23]:
def create_npz_dataset(df):
    x = []
    y_keys = []
    # Extract columns
    class_names = df["Class"].values
    file_paths = df["File Path"].values

    # Process images
    x = []
    y_keys = []

    for file_path, class_name in tqdm(zip(file_paths, class_names), desc="Processing images", total=len(file_paths)):
        images = np.load(file_path)['kernel_pics'].astype('uint8')
        y = [class_name] * images.shape[0]
        x.extend(images)
        y_keys.extend(y)

    x = np.array(x)
    y_keys = np.array(y_keys)

    return x, y_keys

In [24]:
X_train, Y_train = create_npz_dataset(train_df)
X_val, Y_val = create_npz_dataset(val_df)
X_test, Y_test = create_npz_dataset(test_df)

Processing images: 100%|██████████| 11/11 [00:20<00:00,  1.83s/it]


In [25]:
np.savez_compressed(f"{PNG_Paddy_7Class}/train_data.npz",x_train=X_train, y_train = Y_train)
np.savez_compressed(f"{PNG_Paddy_7Class}/val_data.npz",x_val=X_val, y_val=Y_val)
np.savez_compressed(f"{PNG_Paddy_7Class}/test_data.npz",x_test=X_test, y_test=Y_test)